# PoDP Ground Truth — Numbers & MIBiG Migration Report

Live report over `data/PoDP/ground_truth.csv`, `strain_pairings.csv` and
`mibig_migration_report.csv`. Every table below — the funnel included — is
recomputed from those files, so re-running this notebook after a rebuild refreshes
the whole report and nothing can drift out of sync.

**Rebuild the inputs with:**

```bash
python scripts/build_ground_truth.py --resolve-gnps --migration-report
```

Reference: **MIBiG 4.0** (3,013 entries) · Source: 76 PoDP JSON descriptors

In [1]:
import collections
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 140)

DATA = Path("../data/PoDP")
REPORTS = DATA / "reports"
gt = pd.read_csv(DATA / "ground_truth.csv")
pairs = pd.read_csv(DATA / "strain_pairings.csv")
mig = pd.read_csv(REPORTS / "mibig_migration_report.csv")

# CSV round-trips booleans as strings; normalise the ones used for filtering
for col in ("metcalf_rankable", "genome_ambiguous", "mibig_local"):
    gt[col] = gt[col].astype(str).str.lower().eq("true")

print(f"ground_truth      {len(gt):>6} rows x {gt.shape[1]} cols")
print(f"strain_pairings   {len(pairs):>6} rows")
print(f"migration report  {len(mig):>6} cited MIBiG accessions")

ground_truth         115 rows x 63 cols
strain_pairings     4966 rows
migration report      74 cited MIBiG accessions


## 1. How the tables are built

```
76 PoDP JSON descriptors
  |
  |-- BGC_MS2_links (optional curation field, present in 33 projects)
  |     |-- tier from evidence:  experimental -> 1
  |     |                        InChIKey match to cited MIBiG -> 2
  |     |                        neither -> 3
  |     |-- genome via MS2_URL -> metabolomics_file -> genome_label
  |     |   ...or, for molecular-family links, via the GNPS component
  |     `-> ground_truth.csv                         one row per link
  |
  `-- genome_metabolome_links (mandatory, all 76 projects)
        `-> strain_pairings.csv       one row per strain-file pairing, untiered
```

Four reference/lookup inputs feed the columns:

| Input | Used for |
|---|---|
| MIBiG 4.0 (cached, re-downloaded on demand) | compound structures, locus, status, retirement |
| rdkit canonical InChIKey | the tier-2 decision |
| GNPS task result files (cached) | genome resolution for molecular-family links |
| `ms2_naming_metadata.csv` | authoritative original -> renamed MS2 filename map |

Files on disk are keyed to these columns:

```
<study_id>/                                    # study_id  (== podp_id unless merged)
  STUDY_MEMBERS.txt                            # only where >1 record shares a folder
  genomes/<genome_accession>.gbk               # genome_accession
  ms2/<massive_id>__<accession>__<original>    # gnps_massive_id, genome_accession,
                                               # + verbatim tail of ms2_run_url
```

PoDP sometimes records one study twice. Where two project records reference an
identical set of MassIVE depositions they share a folder named for whichever holds
the majority of links (`study_role = main`); the other is `merged`. **`podp_id` is
never overwritten** — `study_id` sits alongside it.

**Tier grades biological evidence only.** Rankability, file status and MIBiG
retirement are orthogonal columns, so a row can be tier 1 and unusable, and both
facts stay visible.

## 2. The funnel

Each gate is applied to the survivors of the previous one, so the `lost` column is
a marginal loss, not a total.

In [2]:
descriptors = sorted(Path("../data/PoDP/json_descriptors").glob("*.json"))

stages = [
    ("PoDP descriptors parsed",        len(descriptors),
     "files in json_descriptors/"),
    ("projects with BGC_MS2_links",    gt["podp_id"].nunique(),
     "optional PoDP curation field"),
    ("  ...folded into shared studies", (gt["study_role"] == "merged").sum(),
     "same PI, identical MassIVE set -> one folder"),
    ("BGC-MS2 links total",            len(gt),
     "ground_truth.csv rows"),
    ("  tier 1 (experimental)",        (gt["tier"] == 1).sum(),
     "verification startswith 'Experimentally validated'"),
    ("  tier 2 (structural match)",    (gt["tier"] == 2).sum(),
     "canonical InChIKey match, full key or connectivity layer"),
    ("  tier 3 (no evidence)",         (gt["tier"] == 3).sum(),
     "retained, never dropped"),
    ("genome via MS2_URL bridge",      gt["genome_source"].eq("ms2_url_bridge").sum(),
     "metabolomics_file -> genome_label"),
    ("genome via GNPS component",      gt["genome_source"].eq("gnps_component").sum(),
     "molecular-family links"),
    ("genome unresolved",              gt["genome_source"].isna().sum(),
     "no bridge fired"),
    ("links in rankable projects",     gt["metcalf_rankable"].sum(),
     ">= 5 declared genomes"),
    ("MS2 run file on disk",           gt["file_ms2_run"].notna().sum(),
     "of the links that name one"),
    ("genome file with sequence",      gt["status_genome"].eq("sequence_ok").sum(),
     "not a WGS/CON stub or an error page"),
    ("data_status = spectrum_pending", gt["data_status"].eq("spectrum_pending").sum(),
     "only the un-extracted MS2 spectrum missing"),
    ("data_status = complete",         gt["data_status"].eq("complete").sum(),
     "all four file types on disk"),
    ("USABLE: tier 1/2 + rankable + genome",
     (gt["tier"].isin([1, 2]) & gt["metcalf_rankable"]
      & gt["status_genome"].eq("sequence_ok")).sum(),
     "has evidence, is scoreable, sequence present"),
    ("strain pairings (untiered)",     len(pairs),
     "strain_pairings.csv -- declared co-origin, not ground truth"),
]

pd.DataFrame(stages, columns=["stage", "n", "what it means"]).set_index("stage")

,n,what it means
stage,,
PoDP descriptors parsed,76,files in json_descriptors/
projects with BGC_MS2_links,33,optional PoDP curation field
...folded into shared studies,5,"same PI, identical MassIVE set -> one folder"
BGC-MS2 links total,115,ground_truth.csv rows
tier 1 (experimental),103,verification startswith 'Experimentally valida...
tier 2 (structural match),8,"canonical InChIKey match, full key or connecti..."
tier 3 (no evidence),4,"retained, never dropped"
genome via MS2_URL bridge,67,metabolomics_file -> genome_label
genome via GNPS component,0,molecular-family links


## 3. Headline

The gap between "has evidence" and "scoreable" is almost entirely **strain depth**
and **broken genome downloads** — not weak biology. Both are fixable.

In [3]:
has_evidence = gt["tier"].isin([1, 2])
usable = has_evidence & gt["metcalf_rankable"] & gt["status_genome"].eq("sequence_ok")
fixable = has_evidence & gt["metcalf_rankable"] & gt["genome_accession"].notna()

pd.DataFrame({
    "n": [len(gt), len(pairs), has_evidence.sum(), usable.sum(), fixable.sum()],
}, index=[
    "Ground truth rows (BGC-spectrum links)",
    "Strain pairings (untiered, separate sheet)",
    "Rows with evidence (tier 1 or 2)",
    "Rows scoreable by Metcalf today",
    "Rows scoreable if genome downloads were fixed",
])

,n
Ground truth rows (BGC-spectrum links),115
"Strain pairings (untiered, separate sheet)",4966
Rows with evidence (tier 1 or 2),111
Rows scoreable by Metcalf today,13
Rows scoreable if genome downloads were fixed,13


## 4. Corpus

`BGC_MS2_links` is an **optional** PoDP curation field. The 43 projects without it
aren't broken — nobody hand-curated a link claim. They hold 2,360 genomes between
them and are the strain-depth pool, not ground truth.

In [4]:
pd.DataFrame({"n": [
    76, gt["podp_id"].nunique(), 76 - gt["podp_id"].nunique(),
    gt["mibig_id"].nunique(), gt["genome_accession"].nunique(),
    gt.groupby("podp_id")["n_genomes_declared"].first().sum(),
]}, index=[
    "Descriptors parsed",
    "Projects with BGC_MS2_links",
    "Projects with no link claim (-> strain pairings only)",
    "Distinct MIBiG accessions cited",
    "Distinct genome accessions referenced",
    "Total genomes declared across those projects",
])

,n
Descriptors parsed,76
Projects with BGC_MS2_links,33
Projects with no link claim (-> strain pairings only),43
Distinct MIBiG accessions cited,74
Distinct genome accessions referenced,35
Total genomes declared across those projects,274


## 5. Tiers

Tier grades **biological evidence only**. Rankability, file status and MIBiG
retirement are separate columns and never affect tier.

| Tier | Definition |
|---|---|
| **1** | Experimentally validated BGC-MS2 link |
| **2** | Canonical InChIKey match to the cited MIBiG compound (full key, or connectivity layer alone) |
| **3** | Neither — no experimental validation, no structural match |

In [5]:
def dist(series, name="n"):
    out = series.value_counts().rename(name).to_frame()
    out["%"] = (100 * out[name] / out[name].sum()).round(0).astype(int)
    return out

dist(gt["tier"]).sort_index()

,n,%
tier,,
1,103,90
2,8,7
3,4,3


In [6]:
# Tier 3 is a missing MIBiG *reference*, never an unknown compound --
# all three rows carry a PoDP SMILES and an IUPAC name.
gt.loc[gt["tier"] == 3, ["podp_id", "mibig_id", "compound_name", "tier_reason"]]

,podp_id,mibig_id,compound_name,tier_reason
96,e26c5e21-e3de-4448-9623-a085cb71eca6.2,BGC0001599,NaN,not experimentally validated; cited MIBiG entr...
98,e26c5e21-e3de-4448-9623-a085cb71eca6.2,BGC0001102,malleilactone,not experimentally validated; InChIKeys differ...
100,e26c5e21-e3de-4448-9623-a085cb71eca6.2,BGC0000572,capistruin,not experimentally validated; MIBiG entry carr...
101,e26c5e21-e3de-4448-9623-a085cb71eca6.2,BGC0001721,ornibactin,not experimentally validated; MIBiG entry carr...


## 6. Evidence composition

`verification` is multi-label, so these sum to more than 100%.

The 33 `similar_MIBiG_number` references point at a **different BGC from a related
strain** — do not treat them as identity downstream.

In [7]:
ev = collections.Counter()
for raw in gt["verification_raw"].dropna():
    for tag in str(raw).split("; "):
        if tag:
            ev[tag] += 1

evidence = pd.Series(ev).sort_values(ascending=False).rename("rows").to_frame()
evidence["% of rows"] = (100 * evidence["rows"] / len(gt)).round(0).astype(int)
evidence

,rows,% of rows
Experimentally validated with NMR and/or detailed MS/MS analysis,93,81
Bioinformatically inferred through comparative analysis with another experimentally defined gene cluster,55,48
"Experimentally validated with knockouts, heterologous expression, or other gene cluster manipulation",39,34
Evidence as indicated in MIBiG,38,33


In [8]:
pd.concat({"link type": dist(gt["link_type"]),
           "MIBiG ref type": dist(gt["mibig_ref_type"])})

n   %
link type      single molecule        66  57
               GNPS molecular family  49  43
MIBiG ref type exact                  82  71
               similar                33  29

## 7. Structural matching

**Tier 2 is decided by rdkit's canonical InChIKey.** A `skeleton` match — the
connectivity layer, first block of the key — counts: the molecular graph agrees and
only stereo annotation differs, which PoDP and MIBiG curate inconsistently.

A SMILES **string** screen is still computed, but purely as a diagnostic. It used to
assign the tier; the cross-tab below is why it no longer does.

In [9]:
pd.concat({"SMILES screen": dist(gt["smiles_match"]),
           "rdkit InChIKey": dist(gt["inchikey_match"])})

n   %
SMILES screen  formula       53  46
               none          27  23
               uncomputable  16  14
               identical     10   9
               partial        6   5
               permutation    3   3
rdkit InChIKey none          48  42
               full          29  25
               skeleton      22  19
               uncomputable  16  14

In [10]:
xt = pd.crosstab(gt["smiles_match"], gt["inchikey_match"], margins=True, margins_name="TOTAL")
order = [c for c in ["full", "skeleton", "none", "uncomputable", "TOTAL"] if c in xt.columns]
rows = [r for r in ["identical", "permutation", "formula", "partial", "none", "TOTAL"] if r in xt.index]
xt.loc[rows, order]

inchikey_match,full,skeleton,none,uncomputable,TOTAL
smiles_match,,,,,
identical,10,0,0,0,10
permutation,3,0,0,0,3
formula,16,22,15,0,53
partial,0,0,6,0,6
none,0,0,27,0,27
TOTAL,29,22,48,16,115


In [11]:
strong = gt["smiles_match"].isin(["identical", "permutation"])
weak = gt["smiles_match"].isin(["formula", "partial"])
comparable = gt["inchikey_match"].ne("uncomputable")

print("Screen reliability, where rdkit could adjudicate")
print(f"  strong calls (identical/permutation): "
      f"{(strong & comparable).sum():>3} rows, "
      f"{(strong & gt['inchikey_match'].eq('full')).sum()} confirmed full")
print(f"  weak calls   (formula/partial)      : "
      f"{(weak & comparable).sum():>3} rows, "
      f"{(weak & gt['inchikey_match'].eq('none')).sum()} REFUTED by rdkit")
print()
agree = (strong == gt["inchikey_match"].eq("full"))[comparable].sum()
print(f"  agree {agree} / disagree {comparable.sum() - agree} / "
      f"uncomputable {(~comparable).sum()}")

Screen reliability, where rdkit could adjudicate
  strong calls (identical/permutation):  13 rows, 13 confirmed full
  weak calls   (formula/partial)      :  59 rows, 21 REFUTED by rdkit

  agree 83 / disagree 16 / uncomputable 16


### Why the string screen was demoted

1. **Its strong signals are exact.** Every `identical` and `permutation` call is a
   true full-InChIKey match — 13 for 13.
2. **Its weak signals are not.** The atom-formula screen is wrong about a third of
   the time, and the 0.60 similarity threshold was refuted on every row it fired
   on. Same molecular formula ≠ same molecule; isomers are common in natural
   products.
3. **It has no false negatives.** rdkit never finds a match the screen called
   `none`, and is never uncomputable where the screen had a verdict — so promoting
   InChIKey cost no coverage.

The screen is retained only so this drift stays measurable.

In [12]:
# Tier 2 membership, and what the InChIKey verdict was for each
t2 = gt[gt["tier"] == 2]
display(t2[["podp_id", "mibig_id", "compound_name", "inchikey_match", "smiles_match"]])

print()
print("Tier 3 -- why each row failed to qualify:")
print(gt.loc[gt["tier"] == 3, ["mibig_id", "tier_reason"]].to_string(index=False))

,podp_id,mibig_id,compound_name,inchikey_match,smiles_match
2,08a05264-7f06-4821-b4ad-bfd4ecb3bd34.2,BGC0000940,desferrioxamine E,full,formula
6,0ff7a302-49af-4130-a440-59e284d4d365.4,BGC0000940,desferrioxamine E,full,formula
8,0ff7a302-49af-4130-a440-59e284d4d365.4,BGC0000296,actinomycin D,skeleton,formula
48,815c513e-8fc3-43a8-88b4-f4df1dd503f7.1,BGC0002055,mycalamide A,skeleton,formula
49,815c513e-8fc3-43a8-88b4-f4df1dd503f7.1,BGC0002057,pateamine,skeleton,formula
50,815c513e-8fc3-43a8-88b4-f4df1dd503f7.1,BGC0002056,peloruside A,skeleton,formula
97,e26c5e21-e3de-4448-9623-a085cb71eca6.2,BGC0000964,burkholdac A,skeleton,formula
99,e26c5e21-e3de-4448-9623-a085cb71eca6.2,BGC0000961,bactobolin,skeleton,formula



Tier 3 -- why each row failed to qualify:
  mibig_id                                                                       tier_reason
BGC0001599    not experimentally validated; cited MIBiG entry absent from this MIBiG release
BGC0001102             not experimentally validated; InChIKeys differ from cited MIBiG entry
BGC0000572 not experimentally validated; MIBiG entry carries no structure to compare against
BGC0001721 not experimentally validated; MIBiG entry carries no structure to compare against


## 8. Genome resolution

The GNPS bridge (15 tasks cached) took resolution from 67/115 to 103/115.

The 12 unresolved rows all got files back from GNPS, but none of those files appear
in the project's declared `metabolomics_file` list. Deliberately **not**
fuzzy-matched — candidates are preserved in `gnps_component_files` for manual
adjudication.

In [13]:
route = gt["genome_source"].fillna("").replace("", "unresolved")
display(dist(route))
print(f"rows mapping to more than one genome (genome_ambiguous): "
      f"{gt['genome_ambiguous'].sum()}  -- expected, a molecular family spans strains")

,n,%
genome_source,,
ms2_url_bridge,67,58
unresolved,48,42


rows mapping to more than one genome (genome_ambiguous): 0  -- expected, a molecular family spans strains


In [14]:
gt.loc[gt["genome_source"].isna(),
       ["podp_id", "mibig_id", "genome_unresolved_reason"]].drop_duplicates()

,podp_id,mibig_id,genome_unresolved_reason
0,03087dd9-2996-4bec-a3cc-88f71babc3a2.1,BGC0002010,gnps_resolution_not_run
1,08a05264-7f06-4821-b4ad-bfd4ecb3bd34.2,BGC0001766,gnps_resolution_not_run
2,08a05264-7f06-4821-b4ad-bfd4ecb3bd34.2,BGC0000940,gnps_resolution_not_run
5,0ff7a302-49af-4130-a440-59e284d4d365.4,BGC0001766,gnps_resolution_not_run
6,0ff7a302-49af-4130-a440-59e284d4d365.4,BGC0000940,gnps_resolution_not_run
7,0ff7a302-49af-4130-a440-59e284d4d365.4,BGC0001830,gnps_resolution_not_run
8,0ff7a302-49af-4130-a440-59e284d4d365.4,BGC0000296,gnps_resolution_not_run
9,0ff7a302-49af-4130-a440-59e284d4d365.4,BGC0000220,gnps_resolution_not_run
11,1b0dccac-5212-4dfd-a9f2-6fa953ab16bd.5,BGC0000632,gnps_resolution_not_run
13,1b0dccac-5212-4dfd-a9f2-6fa953ab16bd.5,BGC0001381,gnps_resolution_not_run


## 9. File availability

`status_genome` classifies what a downloaded record **actually contains**, not
whether a file exists. A WGS/CON master record has a header and no contigs — it is
not a genome.

Nothing extracts single MS2 spectra to their own file yet, so that kind is missing on
every row by construction — hence the separate `spectrum_pending` status.

In [15]:
pd.concat({"genome file status": dist(gt["status_genome"]),
           "data_status": dist(gt["data_status"]),
           "missing_files": dist(gt["missing_files"].fillna("(none)"))})

n   %
genome file status sequence_ok                      62  54
                   absent                           48  42
                   metadata_only_no_contigs          5   4
data_status        spectrum_pending                 58  50
                   incomplete                       57  50
missing_files      ms2_spectrum                     58  50
                   genome|ms2_run|ms2_spectrum      45  39
                   genome|ms2_spectrum               5   4
                   bgc|ms2_spectrum                  3   3
                   bgc|genome|ms2_run|ms2_spectrum   3   3
                   ms2_run|ms2_spectrum              1   1

## 10. Attrition cascade — where the rows go

Order matters: each gate is applied to the survivors of the previous one.

In [16]:
gates = [
    ("All ground truth rows",              pd.Series(True, index=gt.index)),
    ("tier 1 or 2 (has evidence)",         gt["tier"].isin([1, 2])),
    ("+ genome resolved to an accession",  gt["genome_accession"].notna()),
    ("+ project has >=5 declared genomes", gt["metcalf_rankable"]),
    ("+ genome file contains sequence",    gt["status_genome"].eq("sequence_ok")),
    ("+ MS2 run file on disk",             gt["file_ms2_run"].notna()),
    ("+ MS2 spectrum extracted",           gt["file_ms2_spectrum"].notna()),
]

alive = pd.Series(True, index=gt.index)
rows = []
for label, gate in gates:
    before = alive.sum()
    alive = alive & gate
    rows.append({"gate": label, "surviving": alive.sum(), "lost": before - alive.sum()})

pd.DataFrame(rows).set_index("gate")

,surviving,lost
gate,,
All ground truth rows,115,0
tier 1 or 2 (has evidence),111,4
+ genome resolved to an accession,67,44
+ project has >=5 declared genomes,13,54
+ genome file contains sequence,13,0
+ MS2 run file on disk,13,0
+ MS2 spectrum extracted,0,13


### Main influences, ranked

**1. Strain depth — the single largest loss.**
Metcalf is a co-occurrence statistic over the strain axis. With one strain the
occurrence matrix has one column, every pair scores identically, nothing ranks.
This is about **Metcalf, not antiSMASH** — antiSMASH runs fine on one genome.
Not fixable by downloading; it is how the studies were designed.

In [17]:
depth = gt.groupby("podp_id")["n_genomes_declared"].first()
band = pd.cut(depth, [0, 1, 4, 19, 10_000],
              labels=["1 genome", "2-4", "5-19", "20+"])
summary = pd.DataFrame({
    "projects": band.value_counts().sort_index(),
    "rows": gt.assign(band=gt["podp_id"].map(band))
              .groupby("band", observed=False).size(),
})
summary

,projects,rows
1 genome,19,41
2-4,7,24
5-19,3,28
20+,4,22


**2. Broken genome downloads — the largest *fixable* loss.**

In [18]:
rankable = gt[gt["tier"].isin([1, 2]) & gt["metcalf_rankable"]]

def blocker(row):
    if pd.isna(row["genome_accession"]):
        return "genome never resolved"
    if row["status_genome"] != "sequence_ok":
        return f"genome file: {row['status_genome']}"
    if pd.isna(row["file_ms2_run"]):
        return "MS2 run not downloaded"
    return "only the MS2 spectrum extraction"

print(f"Of the {len(rankable)} rankable tier-1/2 rows, what blocks them:\n")
print(rankable.apply(blocker, axis=1).value_counts().to_string())

Of the 46 rankable tier-1/2 rows, what blocks them:

genome never resolved               33
only the MS2 spectrum extraction    13


In [19]:
to_fetch = rankable.loc[rankable["genome_accession"].notna(), "genome_accession"]
have = rankable.loc[rankable["status_genome"].eq("sequence_ok"), "genome_accession"]

print("THE COUNTERFACTUAL THAT MATTERS")
print(f"  usable today                              {int(usable.sum()):>4}")
print(f"  usable if genome downloads were fixed     {len(rankable[rankable['genome_accession'].notna()]):>4}")
print(f"  distinct genome accessions to re-fetch    {to_fetch.nunique():>4}")
print(f"  ...of which already have sequence         {have.nunique():>4}")

THE COUNTERFACTUAL THAT MATTERS
  usable today                                13
  usable if genome downloads were fixed       13
  distinct genome accessions to re-fetch      12
  ...of which already have sequence           12


## 11. Per-project breakdown

The four deepest projects hold 22 rows and have **2 usable genome files between
them**. That is the highest-value download target.

In [20]:
per = (gt.assign(seq_ok=gt["status_genome"].eq("sequence_ok"),
                 unresolved=gt["genome_source"].isna())
         .groupby("podp_id")
         .agg(genomes=("n_genomes_declared", "first"),
              rows=("tier", "size"),
              T1=("tier", lambda s: (s == 1).sum()),
              T2=("tier", lambda s: (s == 2).sum()),
              T3=("tier", lambda s: (s == 3).sum()),
              seq_ok=("seq_ok", "sum"),
              unresolved=("unresolved", "sum"))
         .sort_values("genomes", ascending=False))
per.index = per.index.str.slice(0, 12)
per

,genomes,rows,T1,T2,T3,seq_ok,unresolved
podp_id,,,,,,,
297c364c-b15,120,9,9,0,0,0,9
0ff7a302-49a,39,5,3,2,0,0,5
4b29ddc3-26d,28,2,2,0,0,0,2
84b56cd3-218,24,6,6,0,0,3,3
a4837d37-1df,11,9,9,0,0,9,0
1b0dccac-521,8,13,13,0,0,1,12
e26c5e21-e3d,7,6,0,2,4,0,6
cc996566-f00,4,5,5,0,0,5,0
f2f2d52e-72e,3,7,7,0,0,7,0


## 12. MIBiG 3.1 → 4.0 migration report

MIBiG 4.0 is vendored at `data/PoDP/mibig_4.0/` (3,013 entries: 2,437 active,
377 retired, 199 pending).

**Retirement does not delete an entry** — 4.0 ships retired records with a stated
reason. Those reasons grade **MIBiG's record**, not the biology: `validation failed`
is MIBiG's own schema validation, and `BGC0002016` (lugdunomycin) carries that flag
while having a full structure, a coordinate-bearing locus and 28 annotated genes.
**Retirement therefore does not affect tier** — it lives in `mibig_status`,
`mibig_quality` and `mibig_retirement_reason`.

In [21]:
for col in ["fate", "status_4.0", "structure_change"]:
    tbl = (mig.assign(**{col: mig[col].fillna("(unchanged)")})
              .groupby(col)
              .agg(accessions=("mibig_id", "size"),
                   rows=("ground_truth_rows", "sum"))
              .sort_values("accessions", ascending=False))
    print(f"\nby {col}:")
    print(tbl.to_string())


by fate:
                  accessions  rows
fate                              
carried_over              65   104
added_in_4.0               4     5
dropped_from_4.0           4     5
absent_from_both           1     1

by status_4.0:
            accessions  rows
status_4.0                  
active              64   103
ABSENT               5     6
retired              5     6

by structure_change:
                  accessions  rows
structure_change                  
(unchanged)               62    96
gained_structure           8    14
lost_structure             4     5


In [22]:
changed = mig[mig["structure_change"].notna()].copy()
changed["compound"] = (changed["compound_4.0"].fillna(changed["compound_3.1"])
                       .astype(str).str.split(";").str[0].str.strip())
changed.sort_values(["structure_change", "ground_truth_rows"], ascending=[True, False])[
    ["mibig_id", "compound", "ground_truth_rows", "structure_change"]
]

,mibig_id,compound,ground_truth_rows,structure_change
62,BGC0001841,detoxin P1,6,gained_structure
56,BGC0001766,salinamide A,2,gained_structure
15,BGC0000463,xantholysin A,1,gained_structure
25,BGC0000952,pristinamycin IA,1,gained_structure
43,BGC0001298,4-Z-annimycin,1,gained_structure
61,BGC0001840,detoxin S1,1,gained_structure
65,BGC0002016,lugdunomycin,1,gained_structure
71,BGC0002057,pateamine,1,gained_structure
72,BGC0002061,nostopeptolide A1,2,lost_structure
27,BGC0000962,barbamide,1,lost_structure


In [23]:
flagged = mig[mig["status_4.0"] != "active"]
flagged[["mibig_id", "ground_truth_rows", "status_4.0",
         "retirement_reason_4.0", "compound_3.1"]]

,mibig_id,ground_truth_rows,status_4.0,retirement_reason_4.0,compound_3.1
15,BGC0000463,1,retired,Entry is spread over multiple contigs,NaN
25,BGC0000952,1,retired,Cluster split over multiple loci.,NaN
27,BGC0000962,1,ABSENT,NaN,barbamide
48,BGC0001476,1,ABSENT,NaN,pseudouridimycin
52,BGC0001599,1,ABSENT,NaN,fragin
56,BGC0001766,2,retired,Duplicate of BGC0001230,NaN
65,BGC0002016,1,retired,validation failed,lugdunomycin
71,BGC0002057,1,retired,Entry is spread over multiple contigs,NaN
72,BGC0002061,2,ABSENT,NaN,nostopeptolide A1; nostopeptolide 1052
73,BGC0002077,1,ABSENT,NaN,NaN


### Net effect of the migration

| | 3.1 | 4.0 |
|---|---|---|
| `uncomputable` rows | 24 | **16** |
| structurally matched rows | 66 | **72** |
| Tier 1 / 2 / 3 | 103 / 9 / 3 | 103 / 9 / 3 |

Tier totals are identical, but that conceals a swap:

- `815c513e` `BGC0002057` — **tier 3 → 2** (pateamine gained a structure)
- `e26c5e21` `BGC0001599` — **tier 2 → 3** (fragin lost one)

20 of 115 rows changed in at least one field. The upgrade is a wash on tier counts
and a clear win on structural comparability.

### Duplicate redirect — resolved

MIBiG 4.0 retires one member of a duplicate pair and names the survivor. **88 such
retirements exist; 80 have an active target.** One affects this table.

Verified before repointing: same organism (*Streptomyces* sp. CNB091), same locus
`BK009377.1`, same salinamide A structure — and `BGC0001230` is the richer record
(8 compounds vs 1, `completeness: complete` vs `unknown`).

The source claim is **not** edited: `mibig_id` keeps what PoDP cited,
`mibig_id_resolved` records what was read.

In [24]:
gt.loc[gt["mibig_redirect"].notna(),
       ["podp_id", "mibig_id", "mibig_id_resolved", "mibig_status",
        "compound_name", "smiles_match", "inchikey_match", "tier"]]

,podp_id,mibig_id,mibig_id_resolved,mibig_status,compound_name,smiles_match,inchikey_match,tier
1,08a05264-7f06-4821-b4ad-bfd4ecb3bd34.2,BGC0001766,BGC0001230,active,salinamide A,formula,full,1
5,0ff7a302-49af-4130-a440-59e284d4d365.4,BGC0001766,BGC0001230,active,salinamide A,formula,full,1


## 14. What we cannot get, and why

Two kinds of gap get conflated easily, so they are coded apart in
`data/PoDP/unavailable_data_inventory.csv`:

* **`mapping_gap`** — the file is on disk, but nothing joins it to the link.
  Fixable in code; no download involved.
* **`file_gap`** — the file is genuinely not in hand.

`recoverability` is the actionable column: `automatable` (route known, not yet
built), `manual` (a human must fetch or adjudicate), `deferred` (reachable,
deliberately not pursued), `unavailable` (no known route).

**Deferred 2026-08-27 — the 7 JGI-only genomes (G11).** IMG Taxon OIDs with no
NCBI equivalent. NPOmix used the same JGI identifiers, so their supplement
offers no shortcut. Study `297c364c` already holds 130 of its 157 genomes, so
these 7 change neither rankability nor NPOmix coverage. Fetching them needs a
JGI login and manual per-genome download. Revisit only if that study's strain
axis turns out to be the binding constraint.


In [25]:
import pandas as pd

inv = pd.read_csv(REPORTS / "unavailable_data_inventory.csv")

show = inv[["gap_id", "file_kind", "gap_type", "reason_code", "recoverability",
            "n_ground_truth_rows", "n_strain_pairing_rows", "n_distinct_items"]]
display(show.style.hide(axis="index"))

print("\nby recoverability (gap classes):")
print(inv["recoverability"].value_counts().to_string())

auto = inv.loc[inv.recoverability == "automatable", "n_ground_truth_rows"].sum()
print(f"\nground-truth rows blocked by work that is automatable: {auto}")
print("-- all of it downstream of the GNPS component bridge and spectrum extraction,")
print("   not of any missing download.")


gap_id,file_kind,gap_type,reason_code,recoverability,n_ground_truth_rows,n_strain_pairing_rows,n_distinct_items
G01,ms2_spectrum,not_attempted,spectrum_extraction_not_run,automatable,115,nan,nan
G02,genome,mapping_gap,gnps_component_bridge_not_run,automatable,48,nan,nan
G03,ms2_run,mapping_gap,no_ms2_url_on_family_link,automatable,48,nan,nan
G04,ms2_run,file_gap,massive_download_failed,manual,1,nan,1.000000
G05,bgc,file_gap,mibig_absent_from_both,unavailable,1,nan,1.000000
G06,bgc,file_gap,mibig_dropped_from_4.0,manual,5,nan,4.000000
G07,genome,file_gap,ncbi_returned_wrong_strain,manual,0,166.000000,21.000000
G08,genome,file_gap,metagenome_scale_assembly,manual,3,2.000000,2.000000
G09,genome,file_gap,no_sequence_after_refetch,manual,2,91.000000,10.000000
G10,genome,file_gap,no_resolution_route,manual,0,76.000000,9.000000



by recoverability (gap classes):
recoverability
manual         6
automatable    3
unavailable    1
deferred       1

ground-truth rows blocked by work that is automatable: 211
-- all of it downstream of the GNPS component bridge and spectrum extraction,
   not of any missing download.


## 13. Method notes

- **Tier is biological evidence only.** Rankability (`metcalf_rankable`), file status
  (`status_genome`, `data_status`) and MIBiG retirement (`mibig_status`) are
  orthogonal columns. A row can be tier 1 and unusable; both facts stay visible.
- **Schema detection is per-file.** MIBiG ≤3.1 nests everything under a `cluster`
  key with `compound`/`chem_struct`; 4.0 is flat with `name`/`structure` and turns
  `loci` into a list. The indexer sniffs each file, so a mixed directory cannot
  half-parse.
- **Nothing is dropped silently.** Rows qualifying for no tier are kept at tier 3
  with a `tier_reason`; unresolved genomes carry `genome_unresolved_reason`.
- **`BGC_ID.strain` is not a strain name** — it holds a nucleotide accession. 0 of 29
  values match any `genome_label`. Never use it as a join key.

## 14. Next actions, in value order

1. **Re-fetch 31 genome accessions properly** — takes usable rows 2 → 36. Assembly
   accessions via NCBI datasets; WGS masters via `-style withparts` or assembly FTP.
   Prioritise `297c364c`, `0ff7a302`, `4b29ddc3`, `84b56cd3`.
2. ~~Promote rdkit InChIKey to the primary tier-2 signal~~ — **done**. Tier 2 is now
   the InChIKey verdict; the partial threshold is diagnostic only. `BGC0001102`
   (malleilactone) correctly fell to tier 3.
3. **Adjudicate the 12 unresolved GNPS links** by hand from `gnps_component_files`.
4. **Decide the MS2 spectrum extraction** — missing on all 115 rows, and the only
   thing standing between 14 rows and `complete`.